In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nft_clean.csv")

df["timestamp_dt"] = pd.to_datetime(df["timestamp_dt"])
df["timestamp"] = pd.to_numeric(df["timestamp"])

print(df.shape)
df.head()

(1491465, 17)


,transaction_hash,block_number,timestamp,nft_address,token_id,from_address,to_address,transaction_value,mint_timestamp,transfers_out_from,transfers_in_from,transfers_out_to,transfers_in_to,num_transitions,timestamp_dt,mint_timestamp_dt,month
0,0xe4a428fa61897bc5c2351adf9ff8b99986545cf0c461...,12545223,1622505626,0x57f1887a8BF19b14fC0dF6Fd9B2acc9Af147eA85,4935411008922098323128075484481109194321925919...,0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5,0xB7dAa94FAa6F4Edf3Cb47F7b2cC19AE1816C5191,6.094421e+15,1.622506e+09,139631,2,0,3,3,2021-06-01 00:00:26,2021-06-01 00:00:26,6
1,0xcad6b8d6d9dd5dbb4880c9ed0b699416f49f9b0ae8cd...,12545224,1622505628,0x57f1887a8BF19b14fC0dF6Fd9B2acc9Af147eA85,5886840813581707472973754352258821231902277047...,0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5,0x95318E316F3e02fe0dd9d01D3f3e094D0a16B738,1.015737e+16,1.622506e+09,139631,2,0,4,4,2021-06-01 00:00:28,2021-06-01 00:00:28,6
2,0xfbd4d38c1e79eee4de54efbb5a8dae24301d1c0320e2...,12545225,1622505630,0x57f1887a8BF19b14fC0dF6Fd9B2acc9Af147eA85,3101487780869816715480719553148382667717514406...,0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5,0x843E66E00f1684926Dd895779802465727D09C82,6.500716e+16,1.622506e+09,139631,2,7,6,2,2021-06-01 00:00:30,2021-06-01 00:00:30,6
3,0x2bdf41371a0f6ed17bb2e36ee02265cd93870230e229...,12545225,1622505630,0x0E3A2A1f2146d86A604adc220b4967A898D7Fe07,179744931,0x6958F5e95332D93D21af0D7B9Ca85B8212fEE0A5,0x79066aE1d08c5B32F1EE9a42597f1e8F0eb6e23b,4.920000e+13,NaN,5113,5113,0,10,6,2021-06-01 00:00:30,NaN,6
4,0x1ad06ef90e1fbb8d9df6f32f2f957f30cbd8fa3c2973...,12545226,1622505648,0x06012c8cf97BEaD5deAe237070F9587f8E7A266d,332089,0xb1690C08E213a35Ed9bAb7B318DE14420FB57d8C,0xEE5a2F7400fd6fA1b44b65531d6902C29F0b20C7,7.000000e+15,NaN,22580,13892,0,1,1,2021-06-01 00:00:48,NaN,6


# INISIALISASI LABEL 

In [3]:
df["is_wash_trading"] = 0

df["rule_self_trade"] = 0
df["rule_seller_buyback"] = 0
df["rule_multi_hop_cycle"] = 0
df["rule_high_pair_count"] = 0

# Rule 2 : Identity Trade / Self Trade

In [4]:
df.loc[
    df["from_address"].str.lower() == df["to_address"].str.lower(),
    "rule_self_trade"
] = 1

df["rule_self_trade"].value_counts()

rule_self_trade
0    1491410
1         55
Name: count, dtype: int64

# Rule 1: Seller Buyback < 30 hari

In [5]:
df = df.sort_values(["nft_address", "token_id", "timestamp"]).reset_index(drop=True)

BUYBACK_WINDOW = 30 * 24 * 60 * 60

for _, group in df.groupby(["nft_address", "token_id"]):
    idxs = group.index.tolist()
    
    for i in range(len(group)):
        seller = group.iloc[i]["from_address"]
        t_sell = group.iloc[i]["timestamp"]
        
        future = group.iloc[i+1:]
        buyback = future[
            (future["to_address"] == seller) &
            ((future["timestamp"] - t_sell) <= BUYBACK_WINDOW)
        ]
        
        if len(buyback) > 0:
            df.loc[idxs[i], "rule_seller_buyback"] = 1
            df.loc[buyback.index, "rule_seller_buyback"] = 1

df["rule_seller_buyback"].value_counts()

rule_seller_buyback
0    1490254
1       1211
Name: count, dtype: int64

# Rule 3: Multi-hop cycle < 7 hari

In [6]:
CYCLE_WINDOW = 7 * 24 * 60 * 60
MAX_HOPS = 5

for _, group in df.groupby(["nft_address", "token_id"]):
    group = group.sort_values("timestamp")
    
    for i in range(len(group)):
        start_wallet = group.iloc[i]["from_address"]
        current_wallet = group.iloc[i]["to_address"]
        start_time = group.iloc[i]["timestamp"]
        path_indices = [group.index[i]]
        
        for j in range(i + 1, min(i + 1 + MAX_HOPS, len(group))):
            row = group.iloc[j]
            
            if row["from_address"] != current_wallet:
                continue
            
            current_wallet = row["to_address"]
            path_indices.append(group.index[j])
            
            if row["timestamp"] - start_time > CYCLE_WINDOW:
                break
            
            if current_wallet == start_wallet:
                df.loc[path_indices, "rule_multi_hop_cycle"] = 1
                break

df["rule_multi_hop_cycle"].value_counts()

rule_multi_hop_cycle
0    1490606
1        859
Name: count, dtype: int64

# Rule 4 : High Transaction Count Per Wallet Pair

In [7]:
pair_cols = ["nft_address", "token_id", "from_address", "to_address"]

pair_count = (
    df.groupby(pair_cols)
    .size()
    .reset_index(name="pair_tx_count")
)

omega = pair_count["pair_tx_count"].quantile(0.95)

print("Threshold omega:", omega)

high_pairs = pair_count[pair_count["pair_tx_count"] > omega][pair_cols]

df = df.merge(
    high_pairs.assign(rule_high_pair_count=1),
    on=pair_cols,
    how="left",
    suffixes=("", "_new")
)

df["rule_high_pair_count"] = df["rule_high_pair_count_new"].fillna(
    df["rule_high_pair_count"]
).fillna(0).astype(int)

df = df.drop(columns=["rule_high_pair_count_new"], errors="ignore")

df["rule_high_pair_count"].value_counts()

Threshold omega: 1.0


rule_high_pair_count
0    1491126
1        339
Name: count, dtype: int64

# Merge Label

In [8]:
rule_cols = [
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count"
]

df["wash_score"] = df[rule_cols].sum(axis=1)

df["wash_score"].value_counts().sort_index()

wash_score
0    1489978
1        669
2        659
3        159
Name: count, dtype: int64

# Checking rasio

In [10]:
df_final = df[
    (df["wash_score"] == 0) |
    (df["wash_score"] >= 2)
].copy()

df_final["label"] = (df_final["wash_score"] >= 2).astype(int)

wash_df = df_final[df_final["label"] == 1]
normal_df = df_final[df_final["label"] == 0]

normal_sampled = normal_df.sample(
    n=len(wash_df) * 50,
    random_state=42
)

df_balanced = pd.concat([wash_df, normal_sampled])
df_balanced = df_balanced.sort_values("timestamp").reset_index(drop=True)

df_balanced["label"].value_counts()

label
0    40900
1      818
Name: count, dtype: int64

In [11]:
output_path = "../data/processed/nft_labeled_rules.csv"

df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: ../data/processed/nft_labeled_rules.csv


In [ ]:
Penelitian menggunakan pendekatan heuristic-based weak labeling karena tidak tersedia ground truth resmi untuk transaksi NFT wash trading.

Empat rule digunakan berdasarkan penelitian terdahulu:

Rule	                                    Referensi
1. Seller Buyback < 30 hari	                Liu et al. (2023)
2. Identity Trade / Self-Trade	            La Morgia (2023), Oh (2024)
3. Multi-hop Cycle < 7 hari	                Oh (2024), Von Wachter (2022)
4. High Transaction Count per Wallet Pair	Niu et al. (2024)

Hasil distribusi rule adalah sebagai berikut:

Rule	                Jumlah Triggered	Persentase
1. Self-Trade	        55	                0.0037%
2. Seller Buyback	    1.211	            0.0812%
3. Multi-hop Cycle	    859	                0.0576%
4. High Pair Count	    339	                0.0227%